# DarkSpot — Competitor Parser
Парсим конкурентов через 2GIS API, считаем коэффициент конкурентности.

In [46]:
!pip install requests pandas numpy folium tqdm -q

In [47]:
import requests
import pandas as pd
import numpy as np
import folium
import time
import json
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import Optional

## Конфиг

In [48]:
TWOGIS_API_KEY = "79b8df35-b109-494f-921e-bf2ebf5636bc"  # https://dev.2gis.ru/
BASE_URL = "https://catalog.api.2gis.com/3.0/items"

## Входные данные от клиента

In [49]:
@dataclass
class ClientInput:
    business_type: str          # тип бизнеса: coffeeshop, nail_salon, tobacco_store и т.д.
    target_audience: list       # ['students', 'office_workers', 'families', ...]
    location_name: str          # название района/адреса для отображения
    lat: float                  # широта центра поиска
    lon: float                  # долгота центра поиска
    search_radius_m: int = 1000 # радиус поиска в метрах


client = ClientInput(
    business_type="coffeeshop",
    target_audience=["students", "office_workers"],
    location_name="Москва, район Хамовники",
    lat=55.7312,
    lon=37.5766,
    search_radius_m=1000
)

## Маппинг бизнес-типов → рубрики 2GIS

In [50]:
BUSINESS_TYPE_MAP = {
    "coffeeshop": {
        "queries": ["кофейня", "кофе", "кафе"],
        "rubrics": ["cafe", "coffee"],
        "direct_keywords": ["кофе", "coffee", "espresso", "specialty", "капучино"],
        "indirect_keywords": ["чай", "выпечка", "десерт", "булочная", "кондитерская"]
    },
    "nail_salon": {
        "queries": ["маникюр", "ногтевой сервис", "nail"],
        "rubrics": ["beauty", "nail"],
        "direct_keywords": ["маникюр", "педикюр", "ногти", "nail", "гель"],
        "indirect_keywords": ["салон красоты", "spa", "брови", "ресницы"]
    },
    "tobacco_store": {
        "queries": ["табак", "кальян", "vape", "сигары"],
        "rubrics": ["tobacco", "hookah"],
        "direct_keywords": ["табак", "сигары", "трубка", "кальян", "вейп", "vape"],
        "indirect_keywords": ["кальянная", "барбершоп"]
    },
    "barbershop": {
        "queries": ["барбершоп", "парикмахерская", "мужская стрижка"],
        "rubrics": ["barbershop", "hair"],
        "direct_keywords": ["барбер", "barbershop", "мужская стрижка", "борода"],
        "indirect_keywords": ["парикмахерская", "салон красоты", "стрижка"]
    },
    "restaurant": {
        "queries": ["ресторан"],
        "rubrics": ["restaurant"],
        "direct_keywords": ["ресторан", "restaurant"],
        "indirect_keywords": ["кафе", "бистро", "столовая", "суши", "пицца"]
    }
}

TARGET_AUDIENCE_WEIGHTS = {
    "students": {"price_sensitivity": 0.8, "preferred_formats": ["кофейня", "fast food", "столовая"]},
    "office_workers": {"price_sensitivity": 0.4, "preferred_formats": ["кофейня", "бизнес-ланч", "ресторан"]},
    "families": {"price_sensitivity": 0.5, "preferred_formats": ["кафе", "ресторан", "пиццерия"]},
    "tourists": {"price_sensitivity": 0.3, "preferred_formats": ["ресторан", "кафе", "кофейня"]},
    "young_adults": {"price_sensitivity": 0.6, "preferred_formats": ["кофейня", "бар", "фастфуд"]}
}

## Парсер 2GIS

In [51]:
def parse_place(raw: dict) -> dict:
    rubrics = raw.get("rubrics", [])
    rubric_names = [r.get("name", "") for r in rubrics]

    point = raw.get("point", {})

    address_str = raw.get("address_name", "")

    reviews = raw.get("reviews", {})
    rating = reviews.get("general_rating", None)
    review_count = reviews.get("general_review_count", 0)

    return {
        "id": raw.get("id"),
        "name": raw.get("name", ""),
        "rubrics": ", ".join(rubric_names),
        "address": address_str,
        "lat": point.get("lat"),
        "lon": point.get("lon"),
        "rating": float(rating) if rating else None,
        "review_count": int(review_count)
    }


def fetch_2gis_places(query: str, lat: float, lon: float, radius: int, api_key: str) -> list:
    results = []
    page = 1

    while True:
        params = {
            "q": query,
            "point": f"{lon},{lat}",
            "radius": radius,
            "page_size": 10,
            "page": page,
            "fields": "items.point,items.address,items.rubrics,items.reviews,items.rating",
            "key": api_key,
            "locale": "ru_RU"
        }

        resp = requests.get(BASE_URL, params=params, timeout=10)
        data = resp.json()

        meta_code = data.get("meta", {}).get("code")
        if meta_code != 200:
            msg = data.get("meta", {}).get("error", {}).get("message", "unknown")
            print(f"[2GIS] '{query}': {msg} (code={meta_code})")
            break

        items = data.get("result", {}).get("items", [])
        if not items:
            break

        results.extend(items)

        total = data.get("result", {}).get("total", 0)
        if len(results) >= total or len(results) >= 200:
            break

        page += 1
        time.sleep(0.3)

    return results


def collect_competitors(client: ClientInput, api_key: str) -> pd.DataFrame:
    btype = BUSINESS_TYPE_MAP.get(client.business_type)
    if not btype:
        raise ValueError(f"Неизвестный тип бизнеса: {client.business_type}")

    raw_items = []
    for query in tqdm(btype["queries"], desc="Запросы к 2GIS"):
        items = fetch_2gis_places(query, client.lat, client.lon, client.search_radius_m, api_key)
        raw_items.extend(items)

    seen_ids = set()
    unique_items = []
    for item in raw_items:
        if item.get("id") not in seen_ids:
            seen_ids.add(item["id"])
            unique_items.append(parse_place(item))

    return pd.DataFrame(unique_items)

## Расчёт коэффициента конкурентности

In [52]:
def haversine(lat1, lon1, lat2, lon2) -> float:
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def category_match_score(row: pd.Series, btype_config: dict) -> float:
    text = (row["name"] + " " + row["rubrics"]).lower()

    direct_hits = sum(1 for kw in btype_config["direct_keywords"] if kw.lower() in text)
    indirect_hits = sum(1 for kw in btype_config["indirect_keywords"] if kw.lower() in text)

    if direct_hits > 0:
        return min(1.0, 0.6 + 0.1 * direct_hits)
    elif indirect_hits > 0:
        return min(0.5, 0.2 + 0.1 * indirect_hits)
    return 0.1


def audience_overlap_score(client: ClientInput) -> float:
    if not client.target_audience:
        return 0.5

    btype = BUSINESS_TYPE_MAP[client.business_type]
    relevant_formats = set(btype.get("direct_keywords", []) + btype.get("indirect_keywords", []))

    overlap = 0
    for aud in client.target_audience:
        aud_config = TARGET_AUDIENCE_WEIGHTS.get(aud, {})
        pref = set(f.lower() for f in aud_config.get("preferred_formats", []))
        overlap += len(pref & relevant_formats)

    return min(1.0, 0.3 + overlap * 0.1)


def rating_score(rating: Optional[float], review_count: int) -> float:
    if rating is None:
        return 0.3

    # чем выше рейтинг и больше отзывов — тем серьёзнее конкурент
    review_weight = min(1.0, np.log1p(review_count) / np.log1p(500))
    return (rating / 5.0) * 0.6 + review_weight * 0.4


def proximity_score(distance_m: float, max_radius_m: int) -> float:
    # ближе = опаснее, линейно убывает
    return max(0.0, 1.0 - (distance_m / max_radius_m))


def compute_competition_coefficient(df: pd.DataFrame, client: ClientInput) -> pd.DataFrame:
    if df.empty:
        print("DataFrame пустой, нет данных для обработки")
        return df

    btype_config = BUSINESS_TYPE_MAP[client.business_type]
    audience_weight = audience_overlap_score(client)

    df = df.copy()

    df["distance_m"] = df.apply(
        lambda r: haversine(client.lat, client.lon, r["lat"], r["lon"]) if pd.notna(r["lat"]) else np.nan,
        axis=1,
        result_type="reduce"
    )

    df["score_category"] = df.apply(lambda r: category_match_score(r, btype_config), axis=1, result_type="reduce")
    df["score_rating"] = df.apply(lambda r: rating_score(r["rating"], r["review_count"]), axis=1, result_type="reduce")
    df["score_proximity"] = df["distance_m"].apply(
        lambda d: proximity_score(d, client.search_radius_m) if pd.notna(d) else 0.0
    )

    WEIGHTS = {"category": 0.45, "proximity": 0.30, "rating": 0.25}

    df["competition_score"] = (
        df["score_category"] * WEIGHTS["category"] +
        df["score_proximity"] * WEIGHTS["proximity"] +
        df["score_rating"] * WEIGHTS["rating"]
    ) * audience_weight

    df["competition_score"] = df["competition_score"].clip(0, 1).round(3)

    def label(score):
        if score >= 0.65: return "🔴 Прямой"
        if score >= 0.40: return "🟡 Косвенный"
        return "🟢 Слабый"

    df["competition_label"] = df["competition_score"].apply(label)

    return df.sort_values("competition_score", ascending=False).reset_index(drop=True)

## Запуск

In [53]:
raw_df = collect_competitors(client, TWOGIS_API_KEY)
print(f"Найдено объектов: {len(raw_df)}")

cols = ["name", "rubrics", "address", "rating", "review_count",
        "distance_m", "competition_score", "competition_label"]

if raw_df.empty:
    print("\nAPI не вернул данных. Запускаем на mock-данных для проверки pipeline...\n")

    MOCK_DATA = [
        {"id": "1", "name": "Кофе Хауз", "rubrics": "Кофейня", "address": "ул. Льва Толстого, 4",
         "lat": 55.7320, "lon": 37.5770, "rating": 4.5, "review_count": 312},
        {"id": "2", "name": "Starbucks", "rubrics": "Кофейня, Кафе", "address": "Комсомольский пр., 15",
         "lat": 55.7305, "lon": 37.5750, "rating": 4.2, "review_count": 890},
        {"id": "3", "name": "Шоколадница", "rubrics": "Кафе, Кондитерская", "address": "ул. Пречистенка, 22",
         "lat": 55.7330, "lon": 37.5790, "rating": 3.9, "review_count": 145},
        {"id": "4", "name": "Пекарня Хлеб", "rubrics": "Пекарня", "address": "ул. Остоженка, 7",
         "lat": 55.7290, "lon": 37.5740, "rating": 4.7, "review_count": 67},
        {"id": "5", "name": "Surf Coffee", "rubrics": "Specialty Coffee, Кофейня", "address": "ул. Зубовская, 3",
         "lat": 55.7345, "lon": 37.5810, "rating": 4.8, "review_count": 203},
    ]
    raw_df = pd.DataFrame(MOCK_DATA)

result_df = compute_competition_coefficient(raw_df, client)
display(result_df[cols].head(20))

Запросы к 2GIS:  33%|███▎      | 1/3 [00:04<00:08,  4.31s/it]

[2GIS] 'кофейня': Length of parameter 'page' should be from 1 to 5 (code=400)


Запросы к 2GIS:  67%|██████▋   | 2/3 [00:08<00:04,  4.11s/it]

[2GIS] 'кофе': Length of parameter 'page' should be from 1 to 5 (code=400)


Запросы к 2GIS: 100%|██████████| 3/3 [00:12<00:00,  4.07s/it]

[2GIS] 'кафе': Length of parameter 'page' should be from 1 to 5 (code=400)
Найдено объектов: 100


,name,rubrics,address,rating,review_count,distance_m,competition_score,competition_label
0,"Surf Coffee x East West, кофейня","Точки кофе, Точки безалкогольных напитков","улица Усачёва, 3",4.8,86,135.311071,0.251,🟢 Слабый
1,"Milk&Beans, кофейня",Кофейни,"улица Усачёва, 2 ст1",5.0,4,42.591370,0.233,🟢 Слабый
2,"Даблби, кофейня",Кофейни,"Оболенский переулок, 9 к1",4.5,19,191.886401,0.222,🟢 Слабый
3,"Cofix, кофейня","Точки кофе, Кондитерские изделия","переулок Хользунова, 6",4.8,42,292.267583,0.220,🟢 Слабый
4,"Skuratov Coffee, кофейня",Кофейни,"Комсомольский проспект, 24 ст1",4.6,74,456.064634,0.219,🟢 Слабый
5,"One Price Coffee, кофейня",Точки кофе,"переулок Хользунова, 1",4.0,27,397.033158,0.214,🟢 Слабый
6,"Abc Coffee Roasters, кофейня",Кофейни,"улица Усачёва, 11и",4.8,76,571.195537,0.211,🟢 Слабый
7,"Stars Coffee, кофейня",Кофейни,"Комсомольский проспект, 28",4.0,53,512.976675,0.207,🟢 Слабый
8,"Правда Кофе, экспресс-кофейня","Точки кофе, Продажа кофе","Комсомольский проспект, 28",4.9,35,488.463215,0.202,🟢 Слабый
9,"Cofix, кофейня","Точки кофе, Кондитерские изделия","Комсомольский проспект, 28",4.5,47,499.195533,0.199,🟢 Слабый


In [54]:

raw = fetch_2gis_places("кофейня", client.lat, client.lon, client.search_radius_m, TWOGIS_API_KEY)
print(f"fetch вернул: {len(raw)} объектов")


params = {
    "q": "кофейня",
    "point": f"{client.lon},{client.lat}",
    "radius": client.search_radius_m,
    "fields": "items.point,items.address,items.rubrics,items.reviews,items.rating",
    "key": TWOGIS_API_KEY,
}
resp = requests.get(BASE_URL, params=params)
data = resp.json()
print(f"\nБез page_size — meta code: {data['meta']['code']}, items: {len(data.get('result', {}).get('items', []))}")


params_bare = {
    "q": "кофейня",
    "point": f"{client.lon},{client.lat}",
    "radius": client.search_radius_m,
    "key": TWOGIS_API_KEY,
}
resp2 = requests.get(BASE_URL, params=params_bare)
data2 = resp2.json()
print(f"Голый запрос — meta code: {data2['meta']['code']}, items: {len(data2.get('result', {}).get('items', []))}")

[2GIS] 'кофейня': Length of parameter 'page' should be from 1 to 5 (code=400)
fetch вернул: 50 объектов

Без page_size — meta code: 200, items: 10
Голый запрос — meta code: 200, items: 10


In [55]:
result_df = compute_competition_coefficient(raw_df, client)

cols = ["name", "rubrics", "address", "rating", "review_count",
        "distance_m", "competition_score", "competition_label"]

result_df[cols].head(20)

,name,rubrics,address,rating,review_count,distance_m,competition_score,competition_label
0,"Surf Coffee x East West, кофейня","Точки кофе, Точки безалкогольных напитков","улица Усачёва, 3",4.8,86,135.311071,0.251,🟢 Слабый
1,"Milk&Beans, кофейня",Кофейни,"улица Усачёва, 2 ст1",5.0,4,42.591370,0.233,🟢 Слабый
2,"Даблби, кофейня",Кофейни,"Оболенский переулок, 9 к1",4.5,19,191.886401,0.222,🟢 Слабый
3,"Cofix, кофейня","Точки кофе, Кондитерские изделия","переулок Хользунова, 6",4.8,42,292.267583,0.220,🟢 Слабый
4,"Skuratov Coffee, кофейня",Кофейни,"Комсомольский проспект, 24 ст1",4.6,74,456.064634,0.219,🟢 Слабый
5,"One Price Coffee, кофейня",Точки кофе,"переулок Хользунова, 1",4.0,27,397.033158,0.214,🟢 Слабый
6,"Abc Coffee Roasters, кофейня",Кофейни,"улица Усачёва, 11и",4.8,76,571.195537,0.211,🟢 Слабый
7,"Stars Coffee, кофейня",Кофейни,"Комсомольский проспект, 28",4.0,53,512.976675,0.207,🟢 Слабый
8,"Правда Кофе, экспресс-кофейня","Точки кофе, Продажа кофе","Комсомольский проспект, 28",4.9,35,488.463215,0.202,🟢 Слабый
9,"Cofix, кофейня","Точки кофе, Кондитерские изделия","Комсомольский проспект, 28",4.5,47,499.195533,0.199,🟢 Слабый


## Статистика по зоне

In [56]:
summary = result_df.groupby("competition_label").agg(
    count=("name", "count"),
    avg_score=("competition_score", "mean"),
    avg_rating=("rating", "mean")
).round(3)

print(f"\n📍 Зона: {client.location_name}")
print(f"🏪 Тип бизнеса: {client.business_type}")
print(f"👥 ЦА: {', '.join(client.target_audience)}")
print(f"📏 Радиус: {client.search_radius_m}м\n")
print(summary)


📍 Зона: Москва, район Хамовники
🏪 Тип бизнеса: coffeeshop
👥 ЦА: students, office_workers
📏 Радиус: 1000м

                   count  avg_score  avg_rating
competition_label                              
🟢 Слабый             100      0.144       4.376


## Карта конкурентов

In [57]:
COLOR_MAP = {
    "🔴 Прямой": "red",
    "🟡 Косвенный": "orange",
    "🟢 Слабый": "green"
}

m = folium.Map(location=[client.lat, client.lon], zoom_start=15)

folium.Circle(
    location=[client.lat, client.lon],
    radius=client.search_radius_m,
    color="blue", fill=True, fill_opacity=0.05
).add_to(m)

folium.Marker(
    location=[client.lat, client.lon],
    popup="Ваша потенциальная точка",
    icon=folium.Icon(color="blue", icon="star")
).add_to(m)

for _, row in result_df.dropna(subset=["lat", "lon"]).iterrows():
    color = COLOR_MAP.get(row["competition_label"], "gray")
    popup_text = (
        f"<b>{row['name']}</b><br>"
        f"Рубрика: {row['rubrics']}<br>"
        f"Адрес: {row['address']}<br>"
        f"Рейтинг: {row['rating']} ({row['review_count']} отзывов)<br>"
        f"Расстояние: {row['distance_m']:.0f}м<br>"
        f"<b>Competition score: {row['competition_score']}</b>"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6 + row["competition_score"] * 8,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

m

## Экспорт

In [58]:
output_path = f"darkspot_{client.business_type}_{client.location_name.replace(' ', '_').replace(',', '')}.csv"
result_df[cols].to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Сохранено: {output_path}")

Сохранено: darkspot_coffeeshop_Москва_район_Хамовники.csv


---
## Дополнительно: mock-режим для тестирования без API ключа

In [59]:
MOCK_DATA = [
    {"id": "1", "name": "Кофе Хауз", "rubrics": "Кофейня", "address": "ул. Льва Толстого, 4",
     "lat": 55.7320, "lon": 37.5770, "rating": 4.5, "review_count": 312},
    {"id": "2", "name": "Starbucks", "rubrics": "Кофейня, Кафе", "address": "Комсомольский пр., 15",
     "lat": 55.7305, "lon": 37.5750, "rating": 4.2, "review_count": 890},
    {"id": "3", "name": "Шоколадница", "rubrics": "Кафе, Кондитерская", "address": "ул. Пречистенка, 22",
     "lat": 55.7330, "lon": 37.5790, "rating": 3.9, "review_count": 145},
    {"id": "4", "name": "Пекарня Хлеб", "rubrics": "Пекарня", "address": "ул. Остоженка, 7",
     "lat": 55.7290, "lon": 37.5740, "rating": 4.7, "review_count": 67},
    {"id": "5", "name": "Surf Coffee", "rubrics": "Specialty Coffee, Кофейня", "address": "ул. Зубовская, 3",
     "lat": 55.7345, "lon": 37.5810, "rating": 4.8, "review_count": 203},
    {"id": "6", "name": "Суши-бар Мияко", "rubrics": "Суши, Японская кухня", "address": "Хамовнический вал, 12",
     "lat": 55.7280, "lon": 37.5760, "rating": 4.1, "review_count": 78},
]

mock_df = pd.DataFrame(MOCK_DATA)
mock_result = compute_competition_coefficient(mock_df, client)
mock_result[cols]

,name,rubrics,address,rating,review_count,distance_m,competition_score,competition_label
0,Кофе Хауз,Кофейня,"ул. Льва Толстого, 4",4.5,312,92.414137,0.244,🟢 Слабый
1,Starbucks,"Кофейня, Кафе","Комсомольский пр., 15",4.2,890,126.863355,0.241,🟢 Слабый
2,Surf Coffee,"Specialty Coffee, Кофейня","ул. Зубовская, 3",4.8,203,458.841259,0.239,🟢 Слабый
3,Шоколадница,"Кафе, Кондитерская","ул. Пречистенка, 22",3.9,145,250.278784,0.167,🟢 Слабый
4,Пекарня Хлеб,Пекарня,"ул. Остоженка, 7",4.7,67,293.845310,0.140,🟢 Слабый
5,Суши-бар Мияко,"Суши, Японская кухня","Хамовнический вал, 12",4.1,78,357.801518,0.129,🟢 Слабый
